In [1]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
import findspark

findspark.init("/Users/i545672/spark3/spark-3.5.5-bin-hadoop3")
findspark.find()

'/Users/i545672/spark3/spark-3.5.5-bin-hadoop3'

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = (
    SparkSession
        .builder
        .appName("aqeDynamicCoalescingApp")
        .master("local[4]")
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.sql.adaptive.enabled", "false")
        .getOrCreate()
)

sc= spark.sparkContext
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/23 21:43:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
def getDataFrameStats(dataFrame, columnName):
    outputDf = (
                    dataFrame
                        # Get partition number for each record
                        .withColumn("Partition Number", spark_partition_id())
                        # Group by patitioon and calculate stats for a column
                        .groupBy("Partition Number")
                        .agg(
                            count("*").alias("Record Count"),
                            min(columnName).alias("Min Column Value " + columnName),
                            max(columnName).alias("Max Column Value " + columnName),
                        )
                        .orderBy("Partition Number")
    )
    return outputDf

In [5]:
yellowTaxisDf = spark.read.option("header", "true").option("inferSchema", "true").csv(
    "./Files/YellowTaxis_202210.csv"
)

print("Number of partitions: ", yellowTaxisDf.rdd.getNumPartitions())

Number of partitions:  4


25/06/23 21:43:45 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [7]:
spark.conf.set("spark.sql.shuffle.partitions", 20)

In [8]:
yellowTaxisGroupedDf = (
    yellowTaxisDf
        .groupBy("VendorId", "payment_type")
        .agg(sum("total_amount"))
)
yellowTaxisGroupedDf.show()

+--------+------------+--------------------+
|VendorId|payment_type|   sum(total_amount)|
+--------+------------+--------------------+
|       1|           2|   3407067.990002185|
|       2|           4|   1103.950000000002|
|       1|           0|   704915.6500000019|
|       1|           4|   73231.73999999865|
|       1|           1|1.7903327639982704E7|
|       6|           0|   279048.8899999997|
|       2|           2|    9225232.37000579|
|       1|           3|  186607.24000000514|
|       2|           0|  2899635.7899999204|
|       2|           1|4.7088665280070014E7|
|       2|           3|   29.71999999999995|
+--------+------------+--------------------+



In [9]:
# Check the number of partitions
print("Partitions after group by = " + str(yellowTaxisGroupedDf.rdd.getNumPartitions()))

# Get dataframe stats
getDataFrameStats(yellowTaxisGroupedDf, "VendorId").show()

Partitions after group by = 20


+----------------+------------+-------------------------+-------------------------+
|Partition Number|Record Count|Min Column Value VendorId|Max Column Value VendorId|
+----------------+------------+-------------------------+-------------------------+
|               1|           2|                        1|                        2|
|               4|           1|                        1|                        1|
|               5|           1|                        1|                        1|
|               7|           1|                        1|                        1|
|               8|           1|                        6|                        6|
|               9|           3|                        1|                        2|
|              17|           1|                        2|                        2|
|              18|           1|                        2|                        2|
+----------------+------------+-------------------------+-------------------

In [10]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

In [11]:
yellowTaxisGroupedDf = (
    yellowTaxisDf
        .groupBy("VendorId", "payment_type")
        .agg(sum("total_amount"))
)

# Check the number of partitions
print("Partitions after group by = " + str(yellowTaxisGroupedDf.rdd.getNumPartitions()))

# Get dataframe stats
getDataFrameStats(yellowTaxisGroupedDf, "VendorId").show()

Partitions after group by = 1


+----------------+------------+-------------------------+-------------------------+
|Partition Number|Record Count|Min Column Value VendorId|Max Column Value VendorId|
+----------------+------------+-------------------------+-------------------------+
|               0|          11|                        1|                        6|
+----------------+------------+-------------------------+-------------------------+

